# Dhara — full-corpus BGE-m3 zero-shot retrieval eval

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. ~15 min.

Upload two files when the second cell asks (or drag them into the file panel first).
**Upload the gzipped corpus** — the 138 MB plain file truncates in Colab's upload widget:

- `corpus_v1.jsonl.gz`  — `data/processed/corpus_v1.jsonl.gz` (~12 MB, 39,484 chunks)
- `probe_questions.jsonl` — `data/processed/probe_questions.jsonl` (552 questions, small)

The plain `corpus_v1.jsonl` also works if you get it up intact (Drive mount, `gdown`).

Outputs written to the working dir (download them from the last cell):

- `bge_m3_corpus.npy` — 39484 × 1024 fp16, the **production dense index artifact**
- `bge_m3_corpus_ids.json`, `bge_m3_query.npy`
- `eval_results.json` — every recall number, machine-readable


In [ ]:
!pip -q install "sentence-transformers>=3.0" rank-bm25

In [ ]:
# --- get the two input files (corpus may be .jsonl or .jsonl.gz) ---
import os, gzip, io
def have_corpus():   return os.path.exists("corpus_v1.jsonl") or os.path.exists("corpus_v1.jsonl.gz")
def have_probes():   return os.path.exists("probe_questions.jsonl")

if not (have_corpus() and have_probes()):
    try:
        from google.colab import files
        files.upload()                 # pick corpus_v1.jsonl.gz  and  probe_questions.jsonl
    except Exception:
        pass
assert have_corpus(), "missing corpus_v1.jsonl(.gz) — upload it to the working dir"
assert have_probes(), "missing probe_questions.jsonl — upload it to the working dir"

def read_jsonl(path):
    op = gzip.open if path.endswith(".gz") else open
    with op(path, "rt", encoding="utf-8") as fh:
        rows = []
        for ln, line in enumerate(fh, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(__import__("json").loads(line))
            except Exception as e:
                raise SystemExit(f"{path} line {ln} is corrupt ({e}). "
                                 f"The upload was truncated — re-upload the .gz.")
    return rows

CORPUS_PATH = "corpus_v1.jsonl.gz" if os.path.exists("corpus_v1.jsonl.gz") else "corpus_v1.jsonl"
print("corpus file:", CORPUS_PATH,
      f"{os.path.getsize(CORPUS_PATH)/1e6:.1f} MB")

In [ ]:
import json, re, time, unicodedata, collections
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

assert torch.cuda.is_available(), "no GPU — Runtime → Change runtime type → T4 GPU"
DEV = "cuda"
KS = [1, 5, 10, 20, 50, 100]

chunks = read_jsonl(CORPUS_PATH)
probes = read_jsonl("probe_questions.jsonl")
assert len(chunks) == 39484, f"expected 39484 chunks, got {len(chunks)} — upload truncated"
assert len(probes) == 552,   f"expected 552 questions, got {len(probes)}"
cid   = [c["chunk_id"]      for c in chunks]
cprov = [c["provision_id"]  for c in chunks]
print(f"{len(chunks)} chunks | {len(probes)} questions "
      f"| corpus lang {collections.Counter(c['language'] for c in chunks)} "
      f"| gold lang {collections.Counter(p['lang_tag'] for p in probes)}")

In [ ]:
# --- BGE-m3 dense embeddings (T4, fp16) ---
m = SentenceTransformer("BAAI/bge-m3", device=DEV)
m.max_seq_length = 512

doc = [f"{(c.get('provision_title_bn') or '')} {c['text_bn']}".strip() for c in chunks]
t = time.time()
C = m.encode(doc, batch_size=128, normalize_embeddings=True,
             convert_to_numpy=True, show_progress_bar=True).astype(np.float16)
print(f"corpus embedded {C.shape} in {time.time()-t:.0f}s")

Qtxt = [p["question_bn"] for p in probes]
Q = m.encode(Qtxt, batch_size=128, normalize_embeddings=True,
             convert_to_numpy=True, show_progress_bar=True).astype(np.float16)

np.save("bge_m3_corpus.npy", C)
json.dump(cid, open("bge_m3_corpus_ids.json", "w"))
np.save("bge_m3_query.npy", Q)
print("saved bge_m3_corpus.npy / bge_m3_corpus_ids.json / bge_m3_query.npy")

In [ ]:
# --- dense top-200 per query (GPU matmul) ---
Cg = torch.tensor(C, device=DEV, dtype=torch.float16)
Qg = torch.tensor(Q, device=DEV, dtype=torch.float16)
dense_rank = []
for i in range(0, len(probes), 64):
    s = Qg[i:i+64] @ Cg.T
    dense_rank.append(torch.topk(s, 200, dim=1).indices.cpu().numpy())
dense_rank = np.vstack(dense_rank)
del Cg, Qg; torch.cuda.empty_cache()
print("dense_rank", dense_rank.shape)

In [ ]:
# --- BM25 (approx of dhara.normalize.content_tokens; rerun with repo code for the paper number) ---
BN = "০১২৩৪৫৬৭৮৯"; DIG = {ord(d): str(i) for i, d in enumerate(BN)}
NON = re.compile(r"[^ঀ-৿0-9a-zA-Z\s]")
STOP = set("""আমি আমার আমাকে আমরা আমাদের তুমি তোমার আপনি আপনার সে তার তাকে তারা তাদের তিনি
এই ঐ ওই সেই কোন কোনো যে যা যাহা কি কী কে কেন এবং ও বা কিংবা অথবা কিন্তু তবে যদি তাহলে তাই সুতরাং
করে করা করতে করিতে হবে হয় হয়েছে হইবে ছিল আছে থাকে এর ের থেকে হতে জন্য সাথে সঙ্গে দিয়ে নিয়ে দ্বারা মাধ্যমে
পর আগে পূর্বে মধ্যে উপর প্রতি বিষয়ে সম্পর্কে অনুযায়ী অনুসারে খুব অনেক বেশি কম সব সকল প্রত্যেক একটি একটা
এখন তখন যখন আবার আরও ইত্যাদি না নেই নয়""".split())
def toks(s):
    s = unicodedata.normalize("NFC", s).replace("\u200c", "").replace("\u200d", "")
    s = s.translate(DIG); s = NON.sub(" ", s).lower()
    return [w for w in s.split() if w and w not in STOP]

from rank_bm25 import BM25Okapi
t = time.time()
bm = BM25Okapi([toks(d) for d in doc])
print(f"bm25 indexed in {time.time()-t:.0f}s")
t = time.time()
bm_rank = np.array([np.argsort(-bm.get_scores(toks(p["question_bn"])))[:200] for p in probes])
print(f"bm25 searched in {time.time()-t:.0f}s | bm_rank {bm_rank.shape}")

In [ ]:
# --- RRF hybrid + provision-level recall@k, overall and by gold language ---
def rrf(*rls, k=60, depth=200):
    sc = collections.defaultdict(float)
    for rl in rls:
        for pos, idx in enumerate(list(rl)[:depth]):
            sc[int(idx)] += 1.0 / (k + pos + 1)
    return [i for i, _ in sorted(sc.items(), key=lambda x: -x[1])]
hybrid_rank = [rrf(dense_rank[i], bm_rank[i]) for i in range(len(probes))]

def first_hit(ranked_chunk_idx, gold_provs):
    for r, ci in enumerate(ranked_chunk_idx, 1):
        if cprov[ci] in gold_provs:
            return r
    return None

def score(get_rank):
    agg = {k: collections.Counter() for k in KS}; tot = collections.Counter()
    for i, p in enumerate(probes):
        tag = p["lang_tag"]; tot[tag] += 1; tot["all"] += 1
        rk = first_hit(get_rank(i), set(p["gold_provision_ids"]))
        for k in KS:
            if rk and rk <= k:
                agg[k][tag] += 1; agg[k]["all"] += 1
    return {g: {f"R@{k}": round(agg[k][g] / tot[g], 3) for k in KS}
            for g in ("all", "english", "bengali", "mixed")}

results = {
    "n": len(probes),
    "by_lang_count": dict(collections.Counter(p["lang_tag"] for p in probes)),
    "bm25":       score(lambda i: bm_rank[i]),
    "bge_m3":     score(lambda i: dense_rank[i]),
    "hybrid_rrf": score(lambda i: hybrid_rank[i]),
}
json.dump(results, open("eval_results.json", "w"), indent=2, ensure_ascii=False)

for name in ("bm25", "bge_m3", "hybrid_rrf"):
    print(f"\n=== {name} ===")
    for g in ("all", "english", "bengali", "mixed"):
        row = results[name][g]
        print(f"  {g:8s} " + "  ".join(f"{k}={v:.2f}" for k, v in row.items()))

In [ ]:
# --- download the artifacts ---
try:
    from google.colab import files
    for f in ["bge_m3_corpus.npy", "bge_m3_corpus_ids.json", "bge_m3_query.npy", "eval_results.json"]:
        files.download(f)
except Exception:
    print("not on Colab — grab these from the file panel:")
    print("  bge_m3_corpus.npy  bge_m3_corpus_ids.json  bge_m3_query.npy  eval_results.json")